# Network Traffic Classification — Transformer-Enhanced LSTM

End-to-end walkthrough: preprocessing -> model build -> train -> evaluate ->
confusion matrix / curves -> baseline comparison -> cross-dataset comparison.

Run this notebook **4 times** (or just edit the config cell and re-run) to
cover both required combinations:

| Run | DATASET_NAME | MODEL_TYPE |
|---|---|---|
|1| cicdarknet2020 | transformer_lstm |
|2| cicdarknet2020 | baseline_lstm |
|3| unsw_nb15 | transformer_lstm |
|4| unsw_nb15 | baseline_lstm |

Then run the final **Compare** cell once to generate the baseline-vs-proposed
and cross-dataset comparison tables/plots required by the project.

In [ ]:
import sys, os
from pathlib import Path

src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root = Path("../").resolve()
os.chdir(project_root)
print("Working dir:", os.getcwd())


## 1 — Configuration
Edit these three lines, then run the rest of the notebook top to bottom.

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
DATASET_PATH = "dataset/cicdarknet2020.parquet"   # or "dataset/unsw_nb15.csv"
DATASET_NAME = "cicdarknet2020"                   # or "unsw_nb15"
MODEL_TYPE   = "transformer_lstm"                 # or "baseline_lstm"

EPOCHS     = 50
BATCH_SIZE = 64
LR         = 1e-3
# ─────────────────────────────────────────────────────────────────────────────


## 2 — Train
Uses `src/train.py`'s `train()` function directly, so behavior exactly
matches the CLI / grading reproduction path.

In [ ]:
from train import train

data = train(
    dataset_path=DATASET_PATH,
    dataset_name=DATASET_NAME,
    model_type=MODEL_TYPE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
)
print("Classes:", list(data["label_encoder"].classes_))
print("Binary  :", data["is_binary"])


## 3 — Training Curves

In [ ]:
%matplotlib inline
from IPython.display import Image, display
display(Image(filename=f"figures/{DATASET_NAME}/{MODEL_TYPE}/training_curves.png"))


## 4 — Evaluate
Uses `src/evaluate.py`'s `evaluate()` function: computes accuracy, precision,
recall, F1, confusion matrix, (and ROC-AUC for binary problems), and saves
everything under `results/<dataset>/<model_type>/`.

In [ ]:
from evaluate import evaluate

metrics = evaluate(
    dataset_path=DATASET_PATH,
    dataset_name=DATASET_NAME,
    model_type=MODEL_TYPE,
)
for k, v in metrics.items():
    if k != "confusion_matrix":
        print(f"{k:15s}: {v}")


## 5 — Confusion Matrix

In [ ]:
cm_path = f"figures/{DATASET_NAME}/{MODEL_TYPE}/confusion_matrix.png"
display(Image(filename=cm_path))


## 6 — ROC Curve (binary datasets only)

In [ ]:
roc_path = Path(f"figures/{DATASET_NAME}/{MODEL_TYPE}/roc_curve.png")
if roc_path.exists():
    display(Image(filename=str(roc_path)))
else:
    print("ROC curve not available (multi-class dataset).")


## 7 — Compare
Run this **after** you have trained+evaluated both `transformer_lstm` and
`baseline_lstm` for at least one dataset (ideally both datasets), by
re-running cells 2–6 above with different `MODEL_TYPE` / `DATASET_NAME`
values each time.

This produces:
- `results/comparison/baseline_vs_proposed.csv` / `.txt` — proposed model vs.
  plain-LSTM baseline (required comparison).
- `results/comparison/dataset_comparison.csv` / `.txt` — proposed model across
  both datasets.
- matching bar charts under `figures/comparison/`.

In [ ]:
from compare_results import compare
compare()
